## Vectors and Probability

A text embedding is a list of numbers that captures a sentence's meaning. Logistic Regression asks that given this embedding what is the probability
this text belongs to a certain class (for example positive and negative reviews)

The model that embedds the text does the hard work by quantising the sentiment in the text. The regression model finds the one direction within this
semantic space that best represents the embedded text

The weight vector (w) is as long as the embeddings in the space. Let's say a text embedding is x, The x.w score measures how far a sentence points along that direction.
If the task is related to sentiment, training the regression model will make w align with the dimensions the embedding model itself uses to encode 'postive' vs 'negative' meaning

Text with similar meanings will be clustered together in the embedding space by the model. If the embedding model is poor, no amount of training of your regression model will improve
results.


### Parameters
z = w.x +b

- x = represents the input data you are analysing, in this case the text that has been embedded
- w = a vector of same length as x, represents the strength and importance of relationship between each feature in x and the final prediction
    a postive weight for a feature increases z whereas a negative weight decreases z, reducing the importance of that feature
- b = A single number added to (w.x). It represents the model's inherent preference ot bias towards 1 outcome before looking at the input data
    This shifts the decision boundary up or down (example is if we assume most emails to not be spam, the bias would be more negative so the model
    assumes a default assumption of 'not spam' until the features prove otherwise)
- z =  Serves as the final score. if z is >0 then the model leans towards class 1 whereas z<0 means the model will lean towards 0. In order for this
    to be represented as a probability (0-1) a sigmoid function is applied to the z score so that a higher z score gets a higher probability score (towards 1 and vice versa)


In [5]:
import numpy as np

In [6]:
def sigmoid(z):
    """
    Compute the sigmoid of z

    Parameters:
    z : A scalar or numpy array of any size.

    Returns:
    s : sigmoid(z)
    """
    s = 1 / (1 + np.exp(-z))
    return s

def predict_probability(X, w, b):
    """
    Compute the probability predictions for our vectors.

    Parameters:
    X : the text embeddings, a numpy array of shape (m, n) where m is the number of samples and n is the number of features.
    w : one number per feature, a numpy array of shape (n,) representing the weights.
    b : scalar representing the bias.

    Returns:
    p : numpy array of shape (m,) containing the predicted probabilities.
    """
    return sigmoid(X @ w + b)





## How it works
The dot prodct for each embedding is added to the bias and the sigmoid function determines where between 0 and 1 that embedding lies in terms of class.

In [ ]:
X = np.array([[0.9, 0.1, 0.4],
                [-0.5, 0.8, -0.2],
                [-0.8, -0.7, -0.3]])

w = np.array([2.0, -1.0, 0.5])
w1 = np.array([0.0, 0.0, 0.0])
b = 0.0

print("Predicted probabilities:", predict_probability(X, w, b))
print("Predicted probabilities with zero weights:", predict_probability(X, w1, b)) 

Predicted probabilities: [0.86989153 0.13010847 0.2592251 ]
Predicted probabilities with zero weights: [0.5 0.5 0.5]


when the weights for each feature are 0, the dot product = 0 and z = (0 x 0) + 0.0
The sigmoid function becomes: $$\frac{1}{1 + e^{-0.0}}$$

Which equals 0.5


## Non-Text Features

This isn't just linked to text, any numerical values that represent meaning can be used

Let's do another example with some context below

Say we want to predict whether a bank loan applicant will default on their loan (Class 1: default, Class: 0 Pays on time)

Here we have a 5 profile embedding for an applicant. Each feature represents (Credit Score tier, Debt-to-Income ratio, Length of Employment, Existing Accounts, Age Group)

- Applicant profile (x): [-0.4, 0.9, -0.6, 0.2, 0.1]

- Model weights (w): [-2.1, 1.8, -1.1, 0.5, -0.1] Here each number represents the strength and importance of the feature being measured (negative weights mean the feature lowers risk whereas a positive weight increases risk).

- Model bias (b): -1.2 The baseline score. The baseline here represents the fact that the bank assumes that most people pay their loans back so b is negative

In the example given the weights correspond to the relationship between that feature and the outcome we are trying to predict

For example:

- A higher credit score indicates less risk of defaulting so the weight is negative 
- A higher debt-to-income ration indicates higher risk of defaulting so the weight is positive


## Applicant Profile x Weight Dot Prouct

| Feature | Value \(x\) | Weight \(w\) | Multiplication \(w.x\) | Business Meaning |
| :--- | :---: | :---: | :---: | :--- |
| **Credit Score** | `-0.4` | `-2.1` | `-0.4 * -2.1 = 0.84` | Low credit score increases default risk |
| **Debt-to-Income** | `0.9` | `1.8` | `0.9 * 1.8 = 1.62` | High debt strongly pushes risk up |
| **Employment** | `-0.6` | `-1.1` | `-0.6 * -1.1 = 0.66` | Short job history increases risk |
| **Existing Accounts** | `0.2` | `0.5` | `0.2 * 0.5 = 0.10` | Minor risk from extra accounts |
| **Age Group** | `0.1` | `-0.2` | `0.1 * -0.2 = -0.02` | Older age minimally lowers risk |

- The dot product here adds up to `3.20`.
- When you then add the bias the  z score is `z = 3.20 + (-1.2)`.
- the final logit score is z = 2.0

In [16]:
X = np.array([-0.4, 0.9, -0.6, 0.2, 0.1]) # applicant features

w = np.array([-2.1, 1.8, -1.1, 0.5, -0.1]) # model weights

b = -1.2 # baseline score for loan payback prediction

print("Predicted probability of loan payback:", predict_probability(X, w, b))

Predicted probability of loan payback: 0.8818430221907305


The final result is that this applicant has a probability of defaulting on their loan of 88.08% based on the current parameters
If the decision threshold is at 70% the system would deny their application

## How features and parameters affect probability scores 

Let's say we have the same applicant but the bank deems that debt-to-income isn't that big of a risk as before.
The new weights could be:

w = [-2.1, 0.7, -1.1, 0.5, -0.1]

Or let' say their is a financial crisis so banks become more scared of bad debt
The baseline now is that more people will likely be at risk of defaulting on their loan

`b = 0.5`

In both scenarios let's say the decision threshold is still 70 %


In [ ]:
#Scenario 1: A new feature has been added to the model, which is the debt-to-income ratio of the applicant. 

X = np.array([-0.4, 0.9, -0.6, 0.2, 0.1]) # applicant features
w = np.array([-2.1, 0.3, -1.1, 0.5, -0.1]) # new weights with new debt to income ratio feature
b = -1.2 # same baseline score for loan payback prediction

#Scenario 2: The debt-to-income ratio feature is updated to what it was before but the bias is adjusted.
X2 = np.array([-0.4, 0.9, -0.6, 0.2, 0.1]) # applicant features
w2 = np.array([-2.1, 1.8, -1.1, 0.5, -0.1]) # new weights with old debt to income ratio feature
b2 = -0.5 # adjusted baseline score for loan payback prediction

print(f" Predicted probability of loan payback with reduced debt-to-income ratio feature:, {predict_probability(X, w, b)}")

print(f" Predicted probability of loan payback with adjusted bias:, {predict_probability(X2, w2, b2)}")

 Predicted probability of loan payback with reduced debt-to-income ratio feature:, 0.6592603884513855
 Predicted probability of loan payback with adjusted bias:, 0.9376141486200057


You can see that in scenario 1, the applicant's probability on defaulting with the new parameters is now ~65% and would be approved by the system
whereas in scenario 2 the applicant's probability on defaulting on their loan is now ~94% (higher than before)